# Fabric ETL portfolio assessment

Assess approved Alteryx, PowerCenter, and IDMC exports in Fabric without starting the web app. The notebook uses the same Python parsers and compiler-based target selection as the web app. It does not execute source workflows, save generated definitions, or deploy Fabric items.

Attach a default Lakehouse and a Fabric Runtime 2.0 environment with the accelerator wheel and approved dependencies. Stage only data approved for processing in OneLake. Run the configuration and preview cells first; then set `CONFIRM_ASSESSMENT = True` to assess the selected input.

In [ ]:
from pathlib import Path

LAKEHOUSE_FILES = Path('/lakehouse/default/Files')
INPUT_PATH = 'etl-assessment/input'  # File or directory relative to Lakehouse Files.
REPORT_FOLDER = 'etl-assessment/reports'
CONFIRM_ASSESSMENT = False
PERSIST_DELTA_TABLES = False
WRITE_REPORTS = False
SOURCE_FILTER = ''  # alteryx, informatica_powercenter, informatica_idmc, or blank.
WORKFLOW_SEARCH = ''  # Path substring, or blank.
DETAIL_PATH = ''  # Exact path from the workflow register, or blank.


## Preview the input

Use a workflow file, a PowerCenter XML export, a supported archive, or a portfolio directory. To assess an extracted IDMC package, select its root containing `manifest.json` and `Explore`. When using a mixed estate directory, keep IDMC packages as ZIP files. Keep inputs unchanged during review and assessment.

In [ ]:
import csv
import json
import sys
from collections import Counter
from uuid import uuid4

if sys.version_info < (3, 13):
    raise RuntimeError('Attach a Fabric Spark environment with Python 3.13 or later.')

from accelerator.application import AssessmentApplication
from accelerator.graph import redact


def lakehouse_path(root, relative):
    if not root.is_dir():
        raise FileNotFoundError('Attach a default Lakehouse before continuing.')
    path = Path(relative)
    if path.anchor or not path.parts or '..' in path.parts:
        raise ValueError('Enter a nonempty path relative to Lakehouse Files, without parent segments.')
    result = root / path
    if not result.resolve().is_relative_to(root.resolve()):
        raise ValueError('The selected path is outside Lakehouse Files.')
    return result


def show_rows(rows, empty_message):
    if rows:
        display(spark.createDataFrame(rows))
    else:
        print(empty_message)


def input_signature(discovered):
    return tuple((item.display_path, item.sha256) for item in discovered.workflows)


assessment = None
source = lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH)
application = AssessmentApplication()
preview = application.preview(source)
preview_signature = input_signature(preview)
print(f'{preview.workflow_count} candidates; {preview.skipped_count} skipped.')
show_rows([
    {'path': item.display_path, 'source': item.source_platform.value, 'bytes': item.size_bytes}
    for item in preview.workflows
], 'No supported assets discovered.')
show_rows([
    {'path': redact(item.path), 'reason': item.reason.value}
    for item in preview.skipped
], 'Nothing skipped during discovery.')


## Assess and review portfolio

Confirm the reviewed selection before running this cell. A compiler-backed target is a recommendation, not proof of successful execution or semantic equivalence.

In [ ]:
assessment = None
if not CONFIRM_ASSESSMENT:
    raise RuntimeError('Review the preview and set CONFIRM_ASSESSMENT = True before assessing.')
if not preview_signature:
    raise ValueError('No supported input to assess.')
if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
    raise ValueError('Input path changed. Run the preview again.')

result = application.assess(source, route_alteryx=True)
if input_signature(result) != preview_signature:
    raise RuntimeError('Input fingerprints changed after preview. Run the preview again.')
assessment = result

print(f'{assessment.workflow_count} assets assessed; {assessment.failed_count} inputs failed.')
for field in ('source_platform', 'complexity_band', 'recommended_target', 'migration_wave'):
    counts = Counter(getattr(item, field) for item in assessment.assessments)
    show_rows([
        {'dimension': field, 'value': str(value), 'assets': count}
        for value, count in sorted(counts.items())
    ], f'No results for {field}.')
show_rows([{'failure': redact(message)} for message in assessment.failures], 'No assessment failures.')


In [ ]:
from collections import Counter
from html import escape


if assessment is None:
    raise RuntimeError('Complete the assessment before reviewing results.')


def workflow_record(item):
    return {
        'path': item.path,
        'source_platform': item.source_platform.value,
        'asset_type': item.asset_type.value,
        'tool_count': item.tool_count,
        'connection_count': item.connection_count,
        'complexity_score': item.complexity_score,
        'complexity_band': item.complexity_band,
        'recommended_target': item.recommended_target,
        'migration_wave': item.migration_wave,
    }


def distribution_panel(title, counts):
    maximum = max(counts.values(), default=1)
    bars = ''.join(
        '<div class="bar-row">'
        f'<span>{escape(str(label))}</span>'
        f'<div class="bar-track"><i style="width:{count / maximum * 100:.1f}%"></i></div>'
        f'<strong>{count}</strong>'
        '</div>'
        for label, count in sorted(counts.items(), key=lambda item: (-item[1], str(item[0])))
    )
    return f'<section class="panel"><h2>{escape(title)}</h2>{bars}</section>'


def render_dashboard(items):
    complexity = Counter(item.complexity_band for item in items)
    targets = Counter(item.recommended_target for item in items)
    waves = Counter(item.migration_wave for item in items)
    platforms = Counter(item.source_platform.value for item in items)
    manual = sum(item.recommended_target == 'Manual rework' for item in items)
    high = complexity.get('High', 0)
    status = 'Conditional proceed' if manual or high else 'Proceed to pilot'
    guidance = (
        f'Review {manual} manual-rework and {high} high-complexity assets before migration.'
        if manual or high
        else 'Confirm bindings and validate a representative pilot before broader migration.'
    )
    cards = ''.join(
        f'<div class="metric"><small>{escape(label)}</small><strong>{value}</strong></div>'
        for label, value in (
            ('Assessed assets', len(items)),
            ('High complexity', high),
            ('Pilot wave', waves.get('Pilot', 0)),
            ('Redesign', waves.get('Redesign', 0)),
        )
    )
    rows = ''.join(
        '<tr>'
        f'<td>{escape(item.path)}</td>'
        f'<td>{item.complexity_score}</td>'
        f'<td><span class="tag">{escape(item.complexity_band)}</span></td>'
        f'<td>{escape(item.recommended_target)}</td>'
        f'<td>{escape(item.migration_wave)}</td>'
        '</tr>'
        for item in items[:8]
    )
    html = f'''
    <style>
      .etl-dashboard {{font-family:Aptos,"Segoe UI",sans-serif;color:#242424;background:#f7f4ef;padding:22px;border-radius:16px}}
      .etl-dashboard * {{box-sizing:border-box}} .hero,.panel,.metric {{background:#fff;border:1px solid #dedede;border-radius:14px}}
      .hero {{border-left:5px solid #b11f4b;padding:18px 20px;margin-bottom:14px}} .hero h1 {{font-size:26px;margin:0 0 7px}}
      .hero p {{margin:4px 0;color:#5c5c5c}} .status {{display:inline-block;background:#f8edf1;color:#9a1a41;padding:5px 10px;border-radius:999px;font-weight:700}}
      .metrics {{display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:12px;margin-bottom:14px}}
      .metric {{padding:15px}} .metric small {{display:block;color:#5c5c5c}} .metric strong {{display:block;font-size:26px;margin-top:6px}}
      .grid {{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px}} .panel {{padding:17px}} .panel h2 {{font-size:17px;margin:0 0 13px}}
      .bar-row {{display:grid;grid-template-columns:minmax(120px,1fr) 2fr 28px;gap:10px;align-items:center;margin:9px 0;font-size:13px}}
      .bar-track {{height:10px;background:#eee;border-radius:99px;overflow:hidden}} .bar-track i {{display:block;height:100%;background:#b11f4b;border-radius:99px}}
      table {{width:100%;border-collapse:collapse;font-size:13px}} th,td {{padding:9px;border-bottom:1px solid #e5e5e5;text-align:left}} th {{background:#f5f5f5}}
      .ranked {{grid-column:1/-1;overflow:auto}} .tag {{background:#f8edf1;color:#9a1a41;padding:3px 7px;border-radius:999px;font-weight:700}}
      @media(max-width:760px) {{.metrics,.grid {{grid-template-columns:1fr 1fr}} .ranked {{grid-column:1/-1}}}}
    </style>
    <div class="etl-dashboard">
      <section class="hero"><span class="status">{escape(status)}</span><h1>Portfolio migration summary</h1><p>{escape(guidance)}</p></section>
      <div class="metrics">{cards}</div>
      <div class="grid">
        {distribution_panel('Technical complexity', complexity)}
        {distribution_panel('Recommended Fabric targets', targets)}
        {distribution_panel('Migration waves', waves)}
        {distribution_panel('Source platforms', platforms)}
        <section class="panel ranked"><h2>Highest technical complexity</h2><table><thead><tr><th>Workflow</th><th>Score</th><th>Complexity</th><th>Target</th><th>Wave</th></tr></thead><tbody>{rows}</tbody></table></section>
      </div>
    </div>'''
    displayHTML(html)


register = sorted(assessment.assessments, key=lambda item: (-item.complexity_score, item.path))
render_dashboard(register)
show_rows([
    workflow_record(item) for item in register
    if (not SOURCE_FILTER or item.source_platform.value == SOURCE_FILTER)
    and WORKFLOW_SEARCH.casefold() in item.path.casefold()
], 'No assessed workflows match the filters.')

if DETAIL_PATH:
    matches = [item for item in register if item.path == DETAIL_PATH]
    if len(matches) != 1:
        raise ValueError('DETAIL_PATH must match exactly one workflow in the register.')
    item = matches[0]
    print(redact(item.recommendation_reason))
    print(f'{len(item.dependencies)} dependency references; {len(item.data_sources)} data source references.')
    show_rows([
        {'tool': redact(tool.name), 'count': tool.count, 'support': tool.support}
        for tool in item.tools
    ], 'No tool inventory for this workflow.')
else:
    print('Set DETAIL_PATH to an exact register path to open workflow findings.')


## Migration plan

Review wave allocation, target distribution, next actions, and evidence expected before advancing each wave. These are technical planning aids, not delivery approval or business-priority decisions.

In [ ]:
if assessment is None:
    raise RuntimeError('Complete the assessment before reviewing the migration plan.')

plan_items = sorted(assessment.assessments, key=lambda item: (item.migration_wave, -item.complexity_score, item.path))
wave_order = ('Wave 1', 'Wave 2', 'Pilot', 'Redesign')
wave_actions = {
    'Wave 1': 'Confirm bindings and establish a repeatable migration pattern.',
    'Wave 2': 'Apply the validated pattern and test representative data.',
    'Pilot': 'Prove complex transformations and cross-engine orchestration.',
    'Redesign': 'Resolve unsupported tools and agree a replacement design.',
}
exit_gates = {
    'Wave 1': ('Connections and identities tested', 'Source bindings reviewed'),
    'Wave 2': ('Functional parity accepted', 'Reusable pattern documented'),
    'Pilot': ('Representative results validated', 'Operational controls proven'),
    'Redesign': ('Replacement semantics approved', 'Owner and rollback path identified'),
}
wave_cards = []
target_cards = []
gate_cards = []
for wave in wave_order:
    scoped = [item for item in plan_items if item.migration_wave == wave]
    targets = sorted({item.recommended_target for item in scoped})
    wave_cards.append(
        f'<article class="wave"><span>{escape(wave)}</span><strong>{len(scoped)} assets</strong><p>{escape(wave_actions[wave])}</p></article>'
    )
    target_cards.append(
        f'<div class="fact"><small>{escape(wave)}</small><strong>{escape(", ".join(targets) or "No assessed assets")}</strong></div>'
    )
    gate_cards.append(
        f'<article class="gate"><h3>{escape(wave)} exit evidence</h3><ul>'
        + ''.join(f'<li>{escape(value)}</li>' for value in exit_gates[wave])
        + '</ul></article>'
    )
backlog_rows = ''.join(
    '<tr>'
    f'<td>{escape(item.path)}</td><td>{escape(item.migration_wave)}</td>'
    f'<td>{escape(item.recommended_target)}</td><td>{item.complexity_score}</td>'
    f'<td>{escape(wave_actions[item.migration_wave])}</td>'
    '</tr>'
    for item in plan_items
)
displayHTML(f'''
<style>
 .plan {{font-family:Aptos,"Segoe UI",sans-serif;color:#242424;background:#f7f4ef;padding:22px;border-radius:16px}}
 .plan * {{box-sizing:border-box}} .plan h1 {{font-size:26px;margin:0 0 6px}} .plan .lead {{color:#5c5c5c;margin:0 0 16px}}
 .waves,.targets,.gates {{display:grid;grid-template-columns:repeat(4,minmax(150px,1fr));gap:12px;margin:14px 0}}
 .wave,.gate,.fact,.backlog {{background:#fff;border:1px solid #dedede;border-radius:14px;padding:15px}}
 .wave {{border-top:4px solid #b11f4b}} .wave span,.fact small {{display:block;color:#9a1a41;font-weight:700}}
 .wave strong,.fact strong {{display:block;font-size:20px;margin:6px 0}} .wave p,.gate li {{color:#5c5c5c;font-size:13px}}
 .backlog {{overflow:auto}} table {{width:100%;border-collapse:collapse;font-size:13px}} th,td {{padding:9px;border-bottom:1px solid #e5e5e5;text-align:left}} th {{background:#f5f5f5}}
 .gate h3 {{font-size:15px;margin:0 0 8px}} .gate ul {{padding-left:18px;margin:0}}
 @media(max-width:850px) {{.waves,.targets,.gates {{grid-template-columns:1fr 1fr}}}}
</style>
<div class="plan"><h1>Evidence-based migration plan</h1><p class="lead">Technical sequencing from static assessment evidence. Validate with a representative pilot.</p>
<div class="waves">{''.join(wave_cards)}</div>
<h2>Fabric targets by wave</h2><div class="targets">{''.join(target_cards)}</div>
<h2>Exit evidence</h2><div class="gates">{''.join(gate_cards)}</div>
<section class="backlog"><h2>Migration backlog and next actions</h2><table><thead><tr><th>Workflow</th><th>Wave</th><th>Target</th><th>Complexity</th><th>Next action</th></tr></thead><tbody>{backlog_rows}</tbody></table></section>
</div>''')


## Technical details

Inspect bounded, redacted source evidence. Connection samples and findings are limited to keep notebook output reviewable. The notebook never executes workflow expressions.

In [ ]:
from accelerator.discovery import DiscoveryService
from accelerator.evidence import etl_asset_evidence, workflow_evidence
from accelerator.etl_ir import SourcePlatform
from accelerator.sources.defaults import create_default_registry

if assessment is None:
    raise RuntimeError('Complete the assessment before inspecting technical details.')

discovery = DiscoveryService()
registry = create_default_registry()
discovered = discovery.discover(source)
evidence_items = []
for index, candidate in enumerate(discovered.workflows):
    with discovery.staged_workflow(candidate) as staged:
        if candidate.source_platform is SourcePlatform.ALTERYX:
            evidence_items.append(workflow_evidence(staged, candidate.display_path, index))
        else:
            detected = registry.detect(staged, candidate.display_path)
            package = registry.parse(staged, detected.match)
            evidence_items.extend(
                etl_asset_evidence(package, asset, candidate.display_path, index)
                for asset in package.assets
            )

node_total = sum(item.node_count for item in evidence_items)
edge_total = sum(item.edge_count for item in evidence_items)
dependency_total = sum(len(item.dependencies) for item in assessment.assessments)
finding_total = sum(len(item.findings) for item in evidence_items)
mapping_counts = Counter(
    tool.support
    for item in assessment.assessments
    for tool in item.tools
    for _ in range(tool.count)
)
mapping_labels = {'dataflow': 'Dataflow mapping', 'notebook': 'Notebook mapping', 'pipeline': 'Pipeline mapping', 'manual': 'Manual review', 'neutral': 'Context only'}
mapping_bars = distribution_panel(
    'Tool-mapping distribution',
    Counter({mapping_labels.get(key, key): value for key, value in mapping_counts.items()}),
)
metrics = ''.join(
    f'<div class="metric"><small>{escape(label)}</small><strong>{value}</strong></div>'
    for label, value in (
        ('Assessed assets', len(assessment.assessments)),
        ('Source components', node_total),
        ('Connections', edge_total),
        ('Dependencies', dependency_total),
        ('Static findings', finding_total),
    )
)
findings = []
for item in evidence_items:
    findings.extend((item.workflow_path, redact(value)) for value in item.findings)
finding_rows = ''.join(
    f'<tr><td>{escape(path)}</td><td>{escape(value)}</td></tr>'
    for path, value in findings[:50]
) or '<tr><td colspan="2">No parser findings.</td></tr>'
dependency_rows = ''.join(
    f'<tr><td>{escape(item.path)}</td><td>{escape(redact(value))}</td></tr>'
    for item in assessment.assessments for value in item.dependencies[:20]
) or '<tr><td colspan="2">No dependency references recorded.</td></tr>'
connection_rows = []
for evidence in evidence_items:
    names = {node.id: node.name or node.label or node.id for node in evidence.nodes}
    connection_rows.extend(
        (evidence.workflow_path, names.get(edge.source, edge.source), names.get(edge.target, edge.target))
        for edge in evidence.edges
    )
connection_html = ''.join(
    f'<tr><td>{escape(path)}</td><td>{escape(redact(start))}</td><td>{escape(redact(end))}</td></tr>'
    for path, start, end in connection_rows[:100]
) or '<tr><td colspan="3">No source connections recorded.</td></tr>'
displayHTML(f'''
<style>
 .technical {{font-family:Aptos,"Segoe UI",sans-serif;color:#242424;background:#f7f4ef;padding:22px;border-radius:16px}}
 .technical * {{box-sizing:border-box}} .technical h1 {{font-size:26px;margin:0 0 6px}} .technical .lead {{color:#5c5c5c;margin:0 0 16px}}
 .technical .metrics {{display:grid;grid-template-columns:repeat(5,minmax(110px,1fr));gap:10px;margin-bottom:14px}}
 .technical .metric,.technical .panel {{background:#fff;border:1px solid #dedede;border-radius:14px;padding:15px}}
 .technical .metric small {{display:block;color:#5c5c5c}} .technical .metric strong {{font-size:23px}}
 .technical .grid {{display:grid;grid-template-columns:1fr 1fr;gap:14px}} .technical .panel {{overflow:auto}}
 .technical table {{width:100%;border-collapse:collapse;font-size:12px}} .technical th,.technical td {{padding:8px;border-bottom:1px solid #e5e5e5;text-align:left;vertical-align:top}} .technical th {{background:#f5f5f5}}
 .technical .wide {{grid-column:1/-1}} .technical .bar-row {{display:grid;grid-template-columns:minmax(120px,1fr) 2fr 28px;gap:10px;align-items:center;margin:9px 0;font-size:13px}}
 .technical .bar-track {{height:10px;background:#eee;border-radius:99px;overflow:hidden}} .technical .bar-track i {{display:block;height:100%;background:#b11f4b;border-radius:99px}}
 @media(max-width:850px) {{.technical .metrics,.technical .grid {{grid-template-columns:1fr 1fr}} .technical .wide {{grid-column:1/-1}}}}
</style>
<div class="technical"><h1>Technical assessment evidence</h1><p class="lead">Bounded, redacted evidence for engineering review. Source expressions are never executed.</p>
<div class="metrics">{metrics}</div><div class="grid">{mapping_bars}
<section class="panel"><h2>Dependencies</h2><table><thead><tr><th>Workflow</th><th>Reference</th></tr></thead><tbody>{dependency_rows}</tbody></table></section>
<section class="panel wide"><h2>Static findings</h2><table><thead><tr><th>Workflow</th><th>Finding</th></tr></thead><tbody>{finding_rows}</tbody></table></section>
<section class="panel wide"><h2>Connection sample</h2><p>Showing {min(len(connection_rows), 100)} of {len(connection_rows)} bounded connections.</p><table><thead><tr><th>Workflow</th><th>From</th><th>To</th></tr></thead><tbody>{connection_html}</tbody></table></section>
</div></div>''')


## Persist normalized assessment tables

Set `PERSIST_DELTA_TABLES = True`, rerun the configuration cell, and run this cell to append an immutable assessment run to eight Delta tables. The current run becomes the latest completed run only after all child tables are written. Values are bounded and redacted; effort estimates, absolute input paths, and raw expressions aren't persisted.

In [ ]:
if PERSIST_DELTA_TABLES:
    from datetime import datetime, timezone
    from hashlib import sha256
    from uuid import uuid4

    from pyspark.sql.types import BooleanType, DoubleType, IntegerType, StringType, StructField, StructType, TimestampType

    from accelerator.capacity import COUNT_KEYS, INFORMATICA_COUNT_KEYS, INFORMATICA_WEIGHTS, WEIGHTS, build_capacity_profile

    if assessment is None or not assessment.assessments:
        raise ValueError('Complete a nonempty assessment before persisting Delta tables.')
    if 'evidence_items' not in globals():
        raise RuntimeError('Run Technical details before persisting Delta tables.')
    if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
        raise ValueError('Input path changed. Preview and assess the new input before persisting.')

    run_id = str(uuid4())
    completed_at = datetime.now(timezone.utc)
    run_label = completed_at.strftime('%Y-%m-%d %H:%M UTC')
    workflow_keys = {
        item.path: sha256(f'{run_id}|{item.path}'.encode()).hexdigest()
        for item in assessment.assessments
    }
    evidence_by_path = {item.workflow_path: item for item in evidence_items}
    candidates_by_path = {item.display_path: item for item in discovered.workflows}

    run_rows = [{
        'RunId': run_id,
        'RunLabel': run_label,
        'StartedAtUtc': completed_at,
        'CompletedAtUtc': completed_at,
        'Status': 'completed',
        'WorkflowCount': assessment.workflow_count,
        'FailedCount': assessment.failed_count,
        'SkippedCount': len(assessment.skipped),
        'RulesVersion': ','.join(sorted({item.rules_version for item in assessment.assessments})),
        'IsLatestCompletedRun': False,
    }]
    workflow_rows = [
        {
            'WorkflowKey': workflow_keys[item.path], 'RunId': run_id, 'WorkflowPath': item.path,
            'SourcePlatform': item.source_platform.value, 'AssetType': item.asset_type.value,
            'NativeId': item.native_id, 'Folder': item.folder, 'ToolCount': item.tool_count,
            'ConnectionCount': item.connection_count, 'ComplexityScore': item.complexity_score,
            'ComplexityBand': item.complexity_band, 'RecommendedTarget': item.recommended_target,
            'MigrationWave': item.migration_wave, 'RecommendationReason': redact(item.recommendation_reason)[:2000],
            'RegistryVersion': item.registry_version,
        }
        for item in assessment.assessments
    ]
    tool_rows = [
        {'RunId': run_id, 'WorkflowKey': workflow_keys[item.path], 'ToolName': redact(tool.name)[:256], 'ToolCount': tool.count, 'SupportCategory': tool.support}
        for item in assessment.assessments for tool in item.tools
    ]
    finding_rows = []
    dependency_rows = []
    node_rows = []
    connection_rows = []
    capacity_rows = []
    for item in assessment.assessments:
        key = workflow_keys[item.path]
        profile = item.capacity_profile
        if profile is None and item.source_platform.value == 'alteryx' and item.path in candidates_by_path:
            with discovery.staged_workflow(candidates_by_path[item.path]) as wf_path:
                profile = build_capacity_profile(wf_path)
        if profile is not None:
            active_weights = INFORMATICA_WEIGHTS if profile.profile_kind == 'informatica' else WEIGHTS
            active_keys = INFORMATICA_COUNT_KEYS if profile.profile_kind == 'informatica' else COUNT_KEYS
            capacity_rows.append({
                'RunId': run_id,
                'WorkflowKey': key,
                'ProfileKind': profile.profile_kind,
                'TargetComponents': ', '.join(item.target_components),
                'ComplexityUnits': float(sum(profile.counts[k] * active_weights[k] for k in active_keys)),
                'DestCount': int(profile.counts.get('dest', 0)),
                'HeavySteps': int(profile.heavy_steps),
                'NotebookSteps': int(profile.notebook_steps),
                'PipelineSteps': int(profile.pipeline_steps),
                'TotalSteps': int(profile.total_steps),
                'CoveredByLinkedMapping': bool(profile.covered_by_linked_mapping),
            })
        finding_rows.append({'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'Recommendation', 'FindingText': redact(item.recommendation_reason)[:2000]})
        finding_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'UnsupportedTool', 'FindingText': redact(value)[:2000]}
            for value in item.unsupported_tools
        )
        dependency_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'DependencyReference': redact(value)[:2000]}
            for value in item.dependencies
        )
        evidence = evidence_by_path.get(item.path)
        if evidence is None:
            continue
        finding_rows.extend(
            {'RunId': run_id, 'WorkflowKey': key, 'FindingType': 'Parser', 'FindingText': redact(value)[:2000]}
            for value in evidence.findings
        )
        node_rows.extend(
            {
                'RunId': run_id, 'WorkflowKey': key, 'NodeId': node.id, 'NodeName': redact(node.name)[:256],
                'NodeLabel': redact(node.label)[:256], 'Plugin': redact(node.plugin)[:512],
                'SupportCategory': node.support, 'Disabled': node.disabled, 'X': node.x, 'Y': node.y,
            }
            for node in evidence.nodes
        )
        connection_rows.extend(
            {
                'RunId': run_id, 'WorkflowKey': key, 'SourceNodeId': edge.source, 'TargetNodeId': edge.target,
                'SourceAnchor': edge.source_anchor, 'TargetAnchor': edge.target_anchor,
            }
            for edge in evidence.edges
        )

    schemas = {
        'etl_assessment_run': StructType([
            StructField('RunId', StringType(), False), StructField('RunLabel', StringType(), False),
            StructField('StartedAtUtc', TimestampType(), False), StructField('CompletedAtUtc', TimestampType(), False),
            StructField('Status', StringType(), False), StructField('WorkflowCount', IntegerType(), False),
            StructField('FailedCount', IntegerType(), False), StructField('SkippedCount', IntegerType(), False),
            StructField('RulesVersion', StringType(), False), StructField('IsLatestCompletedRun', BooleanType(), False),
        ]),
        'etl_workflow_assessment': StructType([
            StructField('WorkflowKey', StringType(), False), StructField('RunId', StringType(), False),
            StructField('WorkflowPath', StringType(), False), StructField('SourcePlatform', StringType(), False),
            StructField('AssetType', StringType(), False), StructField('NativeId', StringType(), True),
            StructField('Folder', StringType(), True), StructField('ToolCount', IntegerType(), False),
            StructField('ConnectionCount', IntegerType(), False), StructField('ComplexityScore', IntegerType(), False),
            StructField('ComplexityBand', StringType(), False), StructField('RecommendedTarget', StringType(), False),
            StructField('MigrationWave', StringType(), False), StructField('RecommendationReason', StringType(), False),
            StructField('RegistryVersion', StringType(), False),
        ]),
        'etl_tool_inventory': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('ToolName', StringType(), False), StructField('ToolCount', IntegerType(), False),
            StructField('SupportCategory', StringType(), False),
        ]),
        'etl_finding': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('FindingType', StringType(), False), StructField('FindingText', StringType(), False),
        ]),
        'etl_dependency': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('DependencyReference', StringType(), False),
        ]),
        'etl_workflow_node': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('NodeId', StringType(), False), StructField('NodeName', StringType(), False),
            StructField('NodeLabel', StringType(), False), StructField('Plugin', StringType(), False),
            StructField('SupportCategory', StringType(), False), StructField('Disabled', BooleanType(), False),
            StructField('X', DoubleType(), True), StructField('Y', DoubleType(), True),
        ]),
        'etl_workflow_connection': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('SourceNodeId', StringType(), False), StructField('TargetNodeId', StringType(), False),
            StructField('SourceAnchor', StringType(), False), StructField('TargetAnchor', StringType(), False),
        ]),
        'etl_capacity_profile': StructType([
            StructField('RunId', StringType(), False), StructField('WorkflowKey', StringType(), False),
            StructField('ProfileKind', StringType(), False), StructField('TargetComponents', StringType(), False),
            StructField('ComplexityUnits', DoubleType(), False), StructField('DestCount', IntegerType(), False),
            StructField('HeavySteps', IntegerType(), False), StructField('NotebookSteps', IntegerType(), False),
            StructField('PipelineSteps', IntegerType(), False), StructField('TotalSteps', IntegerType(), False),
            StructField('CoveredByLinkedMapping', BooleanType(), False),
        ]),
    }
    records = {
        'etl_workflow_assessment': workflow_rows, 'etl_tool_inventory': tool_rows,
        'etl_finding': finding_rows, 'etl_dependency': dependency_rows,
        'etl_workflow_node': node_rows, 'etl_workflow_connection': connection_rows,
        'etl_capacity_profile': capacity_rows,
    }
    written = []
    try:
        for table_name, rows in records.items():
            spark.createDataFrame(rows, schemas[table_name]).write.format('delta').mode('append').saveAsTable(table_name)
            written.append(table_name)
        spark.createDataFrame(run_rows, schemas['etl_assessment_run']).write.format('delta').mode('append').saveAsTable('etl_assessment_run')
        written.append('etl_assessment_run')
        spark.sql(
            "UPDATE etl_assessment_run SET IsLatestCompletedRun = "
            f"CASE WHEN RunId = '{run_id}' THEN true ELSE false END WHERE Status = 'completed'"
        )
    except Exception:
        for table_name in reversed(written):
            spark.sql(f"DELETE FROM {table_name} WHERE RunId = '{run_id}'")
        raise
    print(f'Persisted run {run_id} to 8 Delta tables; marked it as the latest completed run.')
else:
    print('Delta persistence is off. Set PERSIST_DELTA_TABLES = True to write normalized tables.')


## Workflow explorer

Expand a workflow to review its assessment summary, bounded source graph, Fabric design interpretation, risks, mappings, and validation checklist.

In [ ]:
if assessment is None:
    raise RuntimeError('Complete the assessment before opening workflows.')
if 'evidence_items' not in globals():
    raise RuntimeError('Run Technical details before opening the Workflow explorer.')


def source_graph_svg(evidence, graph_index):
    nodes = evidence.nodes[:80]
    if not nodes:
        return '<p>No bounded source graph is available.</p>'
    use_source = all(node.x is not None and node.y is not None for node in nodes)
    if use_source:
        min_x = min(node.x for node in nodes if node.x is not None)
        min_y = min(node.y for node in nodes if node.y is not None)
        max_x = max(node.x for node in nodes if node.x is not None)
        scale = min(1.4, 800 / max(max_x - min_x, 1))
        positions = {
            node.id: ((node.x - min_x) * scale + 90, (node.y - min_y) * scale + 60)
            for node in nodes
            if node.x is not None and node.y is not None
        }
    else:
        positions = {node.id: (90 + (index % 5) * 180, 60 + (index // 5) * 100) for index, node in enumerate(nodes)}
    width = max(x for x, _ in positions.values()) + 100
    height = max(y for _, y in positions.values()) + 70
    marker = f'arrow-{graph_index}'
    edges = ''.join(
        f'<path d="M {positions[edge.source][0]} {positions[edge.source][1] + 24} L {positions[edge.target][0]} {positions[edge.target][1] - 24}" marker-end="url(#{marker})"/>'
        for edge in evidence.edges[:200]
        if edge.source in positions and edge.target in positions
    )
    boxes = ''.join(
        f'<g class="{escape(node.support)}"><title>{escape(node.name)} | {escape(node.support)}</title>'
        f'<rect x="{positions[node.id][0] - 72}" y="{positions[node.id][1] - 24}" width="144" height="48" rx="8"/>'
        f'<text x="{positions[node.id][0]}" y="{positions[node.id][1] + 4}">{escape((node.label or node.name or node.id)[:20])}</text></g>'
        for node in nodes
    )
    return (
        f'<div class="graph"><svg viewBox="0 0 {width} {height}" role="img" aria-label="Source workflow graph">'
        f'<defs><marker id="{marker}" viewBox="0 0 8 8" markerWidth="6" markerHeight="6" refX="8" refY="4" orient="auto"><path d="M0 0 8 4 0 8Z"/></marker></defs>'
        f'<g class="edges">{edges}</g>{boxes}</svg></div>'
    )


evidence_by_path = {item.workflow_path: item for item in evidence_items}
selected_item = None
selected_evidence = None
graph = '<p>No assessed workflows available.</p>'
risk_html = '<li>No assessed workflows available.</li>'
mappings = '<tr><td colspan="3">No assessed workflows available.</td></tr>'
if not register:
    print('No assessed workflows are available to explore.')
else:
    selected_path = DETAIL_PATH or register[0].path
    selected_matches = [item for item in register if item.path == selected_path]
    if len(selected_matches) != 1:
        raise ValueError('DETAIL_PATH must match exactly one workflow in the register.')
    selected_item = selected_matches[0]
    selected_index = register.index(selected_item)
    selected_evidence = evidence_by_path.get(selected_path)
    findings = list(selected_evidence.findings[:5]) if selected_evidence else []
    risks = [f'Unsupported tool: {value}' for value in selected_item.unsupported_tools[:5]]
    risks.extend(f'Dependency: {redact(value)}' for value in selected_item.dependencies[:5])
    risks.extend(redact(value) for value in findings)
    risk_html = ''.join(f'<li>{escape(value)}</li>' for value in risks) or '<li>No risks from the bounded static checks. Other risks may remain.</li>'
    mappings = ''.join(
        f'<tr><td>{escape(tool.name)}</td><td>{tool.count}</td><td>{escape(tool.support)}</td></tr>'
        for tool in selected_item.tools
    ) or '<tr><td colspan="3">No tool mappings recorded.</td></tr>'
    graph = source_graph_svg(selected_evidence, selected_index) if selected_evidence else '<p>Source graph not available.</p>'
    displayHTML(f'''
<style>
 .explorer {{font-family:Aptos,"Segoe UI",sans-serif;color:#242424;background:#f7f4ef;padding:22px;border-radius:16px}}
 .explorer * {{box-sizing:border-box}} .explorer h1 {{font-size:26px;margin:0 0 6px}} .explorer>.lead {{color:#5c5c5c;margin:0 0 16px}}
 .overview {{display:grid;grid-template-columns:repeat(4,1fr);gap:10px}} .overview div {{background:#fff;border:1px solid #e1e1e1;border-radius:12px;padding:13px}}
 .overview small,.overview strong {{display:block}} .overview strong {{font-size:18px;margin-top:4px}} @media(max-width:850px) {{.overview {{grid-template-columns:1fr 1fr}}}}
</style>
<div class="explorer"><h1>Workflow summary</h1><p class="lead">Selected workflow: {escape(selected_item.path)}. Change DETAIL_PATH and rerun the workflow cells to inspect another workflow.</p>
<section class="overview"><div><small>Complexity score</small><strong>{selected_item.complexity_score}</strong></div><div><small>Target</small><strong>{escape(selected_item.recommended_target)}</strong></div><div><small>Wave</small><strong>{escape(selected_item.migration_wave)}</strong></div><div><small>Dependencies</small><strong>{len(selected_item.dependencies)}</strong></div></section></div>''')


In [ ]:
if selected_item is not None:
    displayHTML(f'''
    <style>.subtab {{font-family:Aptos,"Segoe UI",sans-serif;background:#f7f4ef;padding:22px;border-radius:16px;color:#242424}} .subtab h1 {{font-size:25px}} .graph {{max-height:520px;overflow:auto;background:#fff;border:1px solid #dedede;border-radius:12px}} .graph svg {{display:block;min-width:100%;height:auto}} .graph .edges path {{stroke:#687485;stroke-width:1.5;fill:none}} .graph marker path {{fill:#687485}} .graph rect {{fill:#fff;stroke:#919191}} .graph text {{font-size:12px;text-anchor:middle;fill:#242424}} .graph .manual rect {{fill:#fff3e0;stroke:#9c5617;stroke-dasharray:4 2}} .graph .notebook rect {{fill:#eef3ff;stroke:#315b9a}} .graph .dataflow rect {{fill:#edf7f1;stroke:#28734a}}</style>
    <div class="subtab"><h1>Source workflow</h1><p>{escape(selected_item.path)} · {'Source coordinates' if selected_evidence and selected_evidence.layout_source == 'source' else 'Derived layout'}</p>{graph}<p>Bounded to 80 components and 200 connections.</p></div>''')


In [ ]:
if selected_item is not None:
    displayHTML(f'''
    <style>.design-tab {{font-family:Aptos,"Segoe UI",sans-serif;background:#f7f4ef;padding:22px;border-radius:16px;color:#242424}} .design-flow {{display:flex;align-items:center;gap:10px;flex-wrap:wrap;background:#fff;border:1px solid #dedede;border-radius:14px;padding:20px}} .design-flow span {{padding:12px;border:1px solid #dedede;border-radius:10px}} .design-flow .primary {{background:#b11f4b;color:#fff;border-color:#b11f4b}}</style>
    <div class="design-tab"><h1>Fabric design</h1><div class="design-flow"><span>Source bindings</span><i>→</i><span>OneLake landing</span><i>→</i><span class="primary">{escape(selected_item.recommended_target)}</span><i>→</i><span>Parity validation</span></div><p>{escape(redact(selected_item.recommendation_reason))}</p><p>This is a target interpretation, not a generated or deployed architecture.</p></div>''')


In [ ]:
if selected_item is not None:
    displayHTML(f'''
    <style>.risk-tab {{font-family:Aptos,"Segoe UI",sans-serif;background:#f7f4ef;padding:22px;border-radius:16px;color:#242424}} .risk-card {{background:#fff;border:1px solid #dedede;border-left:5px solid #b11f4b;border-radius:14px;padding:18px}} .risk-card li {{margin:8px 0;color:#5c5c5c}}</style>
    <div class="risk-tab"><h1>Risk assessment</h1><section class="risk-card"><h2>{escape(selected_item.path)}</h2><ul>{risk_html}</ul></section></div>''')


In [ ]:
if selected_item is not None:
    displayHTML(f'''
    <style>.validation-tab {{font-family:Aptos,"Segoe UI",sans-serif;background:#f7f4ef;padding:22px;border-radius:16px;color:#242424}} .check {{background:#fff;border:1px solid #dedede;border-radius:14px;padding:18px}} .check li {{margin:10px 0}} .notice {{color:#9a1a41;font-weight:700}}</style>
    <div class="validation-tab"><h1>Validation</h1><section class="check"><p class="notice">Runtime validation has not been performed.</p><ul><li>Confirm source and destination bindings and permissions.</li><li>Compare schemas, row counts, null handling, and business rules.</li><li>Test representative data and exception paths.</li><li>Validate operational controls, monitoring, and rollback before promotion.</li></ul></section></div>''')


In [ ]:
if selected_item is not None:
    displayHTML(f'''
    <style>.mapping-tab {{font-family:Aptos,"Segoe UI",sans-serif;background:#f7f4ef;padding:22px;border-radius:16px;color:#242424}} .mapping-tab table {{width:100%;border-collapse:collapse;background:#fff;border:1px solid #dedede;border-radius:14px;overflow:hidden}} .mapping-tab th,.mapping-tab td {{padding:10px;border-bottom:1px solid #e5e5e5;text-align:left}} .mapping-tab th {{background:#f5f5f5}}</style>
    <div class="mapping-tab"><h1>Tool mappings</h1><p>{escape(selected_item.path)}</p><table><thead><tr><th>Tool</th><th>Count</th><th>Mapping</th></tr></thead><tbody>{mappings}</tbody></table></div>''')


## Export reports

Set `WRITE_REPORTS = True` and rerun this cell to save a new CSV and JSON report to the Lakehouse. Reports omit estimated effort, source configuration, and absolute input paths. File names, asset identifiers, and hashes remain in the JSON so the offline HTML can match supported inputs.

In [ ]:
def csv_value(value):
    if isinstance(value, str) and (value.startswith(('\t', '\r', '\n')) or value.lstrip().startswith(('=', '+', '-', '@'))):
        return "'" + value
    return value


if WRITE_REPORTS:
    if assessment is None or not assessment.assessments:
        raise ValueError('Complete a nonempty assessment before exporting.')
    if source != lakehouse_path(LAKEHOUSE_FILES, INPUT_PATH):
        raise ValueError('Input path changed. Preview and assess the new input before exporting.')
    output_root = lakehouse_path(LAKEHOUSE_FILES, REPORT_FOLDER)
    if output_root.resolve().is_relative_to(source.resolve()):
        raise ValueError('Choose a report folder outside the input path.')
    output = output_root / f'run-{uuid4().hex}'
    output.mkdir(parents=True, exist_ok=False)
    ordered = sorted(assessment.assessments, key=lambda item: (-item.complexity_score, item.path))
    rows = [workflow_record(item) for item in ordered]
    report = {
        'workflow_count': assessment.workflow_count,
        'failed_count': assessment.failed_count,
        'workflows': [item.model_dump(mode='json', include={'display_path', 'sha256'}) for item in assessment.workflows],
        'assessments': [
            {**row, **item.model_dump(mode='json', include={'native_id', 'folder', 'rules_version', 'registry_version'})}
            for row, item in zip(rows, ordered, strict=True)
        ],
    }
    with (output / 'assessment.csv').open('x', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows({key: csv_value(value) for key, value in row.items()} for row in rows)
    with (output / 'assessment.json').open('x', encoding='utf-8') as stream:
        json.dump(report, stream, indent=2)
    print(f'Saved assessment.csv and assessment.json in Files/{output.relative_to(LAKEHOUSE_FILES).as_posix()}')
else:
    print('Report writing is off. Set WRITE_REPORTS = True to export.')
